# DP Single-Phase State-Space Nodal Averaged Voltage Source Inverter Model

This notebook validates the DP (dynamic phasor) single-phase `AvVoltSourceInverterStateSpace` component in the same small grid-connected converter example as the EMT reference notebook (`EMT_Ph3_AvVoltSourceInverterStateSpace.ipynb`).

The simulation consists of:

1. an SP-domain power-flow initialization, shared by both dynamic simulations,
2. an EMT-domain dynamic simulation, run here as the cross-domain reference,
3. a DP-domain dynamic simulation of the same circuit and parameters,
4. a resistive load disturbance connected through a switch in both dynamic simulations.

The notebook checks that:

- invalid parameter sets raise exceptions,
- the DP simulation starts close to the intended steady state,
- after the load-switching disturbance, the DP converter returns close to the active/reactive power references,
- the DP PLL frequency returns close to the nominal angular frequency,
- the DP q-axis capacitor voltage converges close to zero,
- the DP PCC voltage remains bounded and settles,
- the DP result tracks the EMT reference closely in both windows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import dpsimpy

sp = dpsimpy.sp
emt = dpsimpy.emt
dp = dpsimpy.dp

sp_ph1 = sp.ph1
emt_ph3 = emt.ph3
dp_ph1 = dp.ph1

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
PowerflowBusType = dpsimpy.PowerflowBusType
Solver = dpsimpy.Solver

## Parameters

The example is parameterized as a 400 V RMS line-to-line low-voltage grid-connected converter, identical to the EMT reference notebook.

The inverter controls filter-side active and reactive power. For the SP-domain power-flow initialization, the corresponding PCC-side power injection is corrected for the resistive loss across `Rc`.

The switched resistor load is intentionally not included in the power flow. It is connected only during the EMT and DP dynamic simulations to create a disturbance.

Both dynamic simulations use the same time step, so the DP result can be compared directly against the EMT result sample by sample, not just against the nominal setpoints.

In [ ]:
time_step = 100e-6
final_time = 1.0
load_switch_time = 0.1

system_frequency = 50.0
system_omega = 2.0 * np.pi * system_frequency

grid_voltage_rms_ll = 400.0

line_resistance = 0.3
line_inductance = 0.1e-3
line_capacitance = 1e-6
line_conductance = 1e-6

load_active_power = 10000.0
switch_open_resistance = 1e12
switch_closed_resistance = 1e-6

lf = 2e-3
cf = 10e-6
rf = 0.2
rc = 0.2

kp_pll = 0.25
ki_pll = 0.2
omega_cutoff = system_omega

p_ref = 10000.0
q_ref = 5000.0

kp_power_ctrl = 0.05
ki_power_ctrl = 0.2

kp_curr_ctrl = 0.25
ki_curr_ctrl = 1.0

sim_name_pf = "DP_Ph1_AvVoltSourceInverterStateSpace_PF"
sim_name_emt = "EMT_Ph3_AvVoltSourceInverterStateSpace_EMT"
sim_name_dp = "DP_Ph1_AvVoltSourceInverterStateSpace_DP"

## Helper functions

In [ ]:
def load_resistance():
    return grid_voltage_rms_ll**2 / load_active_power


def pcc_power_from_filter_power_reference(p_filter_ref, q_filter_ref):
    v_pcc_peak_phase = np.sqrt(2.0 / 3.0) * grid_voltage_rms_ll

    if abs(rc) < 1e-12 or v_pcc_peak_phase < 1e-9:
        return p_filter_ref, q_filter_ref

    # With peak phase phasors:
    #
    # P_filter = P_pcc + Rc / (1.5 * |U|^2) * (P_pcc^2 + Q_pcc^2)
    # Q_filter = Q_pcc
    q_pcc_ref = q_filter_ref
    a = rc / (1.5 * v_pcc_peak_phase**2)

    discriminant = 1.0 + 4.0 * a * (p_filter_ref - a * q_pcc_ref**2)
    if discriminant < 0.0:
        raise RuntimeError(
            "No feasible PCC power found for the requested filter-side power "
            "reference, Rc, and PCC voltage estimate."
        )

    sqrt_disc = np.sqrt(discriminant)
    p1 = (-1.0 + sqrt_disc) / (2.0 * a)
    p2 = (-1.0 - sqrt_disc) / (2.0 * a)

    # Select the root close to p_filter_ref, which is the physically relevant
    # branch for small Rc.
    p_pcc_ref = p1 if abs(p1 - p_filter_ref) < abs(p2 - p_filter_ref) else p2

    return p_pcc_ref, q_pcc_ref


def signal(df, name):
    cols = [c for c in df.columns if c == name or c.startswith(name + "_")]
    if not cols:
        raise KeyError(f"No columns found for signal '{name}'.")
    return df[cols].to_numpy()


def scalar_signal(df, name):
    values = signal(df, name)
    if values.shape[1] != 1:
        raise ValueError(f"Signal '{name}' is not scalar.")
    return values[:, 0]


def complex_signal(df, name):
    return (df[name + ".re"] + 1j * df[name + ".im"]).to_numpy()


def time_vector(df):
    return df.iloc[:, 0].to_numpy()


def rms(values):
    values = np.asarray(values)
    return np.sqrt(np.mean(values**2))


def window(df, start, end):
    time = time_vector(df)
    return (time >= start) & (time <= end)


def phase_vector_magnitude(values_abc):
    return np.linalg.norm(values_abc, axis=1)


def envelope_magnitude(values_complex):
    # DP::Ph1's envelope magnitude is already on the same scale as
    # `phase_vector_magnitude`: both equal the RMS line-to-line voltage
    # directly for a balanced three-phase signal (sqrt(3/2) * peak-phase
    # cancels against the sqrt(2/3) peak-phase-to-RMS-line-line factor).
    return np.abs(values_complex)

## Build and run the SP-domain power flow

The power-flow case initializes the grid and PCC voltage without the switched load. Its result seeds both the EMT reference simulation and the DP simulation.

In [ ]:
def run_power_flow():
    Logger.set_log_dir("logs/" + sim_name_pf)

    n_grid = sp.SimNode("nGrid", PhaseType.Single)
    n_pcc = sp.SimNode("nPcc", PhaseType.Single)

    slack = sp_ph1.NetworkInjection("Slack")
    slack.set_parameters(grid_voltage_rms_ll)
    slack.set_base_voltage(grid_voltage_rms_ll)
    slack.modify_power_flow_bus_type(PowerflowBusType.VD)

    line = sp_ph1.PiLine("Line")
    line.set_parameters(
        line_resistance,
        line_inductance,
        line_capacitance,
        line_conductance,
    )
    line.set_base_voltage(grid_voltage_rms_ll)

    p_pcc_ref, q_pcc_ref = pcc_power_from_filter_power_reference(p_ref, q_ref)

    # Negative load = injected P/Q at PCC.
    inverter_injection = sp_ph1.Load("INV_SSN_PLL")
    inverter_injection.set_parameters(-p_pcc_ref, -q_pcc_ref, grid_voltage_rms_ll)
    inverter_injection.modify_power_flow_bus_type(PowerflowBusType.PQ)

    slack.connect([n_grid])
    line.connect([n_grid, n_pcc])
    inverter_injection.connect([n_pcc])

    system = SystemTopology(
        system_frequency,
        [n_grid, n_pcc],
        [slack, line, inverter_injection],
    )

    logger = Logger(sim_name_pf)
    logger.log_attribute("v_grid_pf", n_grid.attr("v"))
    logger.log_attribute("v_pcc_pf", n_pcc.attr("v"))

    time_step_pf = final_time
    final_time_pf = final_time + time_step_pf

    sim = Simulation(sim_name_pf)
    sim.set_system(system)
    sim.set_time_step(time_step_pf)
    sim.set_final_time(final_time_pf)
    sim.set_domain(Domain.SP)
    sim.set_solver(Solver.NRP)
    sim.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
    sim.do_init_from_nodes_and_terminals(False)
    sim.add_logger(logger)
    sim.run()

    return system


system_pf = run_power_flow()

## Build and run the EMT-domain reference simulation

This is the same three-phase averaged-inverter scenario as `EMT_Ph3_AvVoltSourceInverterStateSpace.ipynb`, run here to produce the cross-domain reference for the DP result below.

In [ ]:
def three_phase_param(value):
    return dpsimpy.Math.single_phase_parameter_to_three_phase(value)


def three_phase_variable(value):
    return dpsimpy.Math.single_phase_variable_to_three_phase(value)


def run_emt_simulation(system_pf):
    Logger.set_log_dir("logs/" + sim_name_emt)

    n_grid = emt.SimNode("nGrid", PhaseType.ABC)
    n_pcc = emt.SimNode("nPcc", PhaseType.ABC)
    n_load = emt.SimNode("nLoad", PhaseType.ABC)

    slack = emt_ph3.NetworkInjection("Slack")
    slack.set_parameters(three_phase_variable(grid_voltage_rms_ll), system_frequency)

    line = emt_ph3.PiLine("Line")
    line.set_parameters(
        three_phase_param(line_resistance),
        three_phase_param(line_inductance),
        three_phase_param(line_capacitance),
        three_phase_param(line_conductance),
    )

    inverter = emt_ph3.AvVoltSourceInverterStateSpace("INV_SSN_PLL")
    inverter.set_parameters(
        lf,
        cf,
        rf,
        rc,
        system_omega,
        kp_pll,
        ki_pll,
        omega_cutoff,
        p_ref,
        q_ref,
        kp_power_ctrl,
        ki_power_ctrl,
        kp_curr_ctrl,
        ki_curr_ctrl,
    )

    load_switch = emt_ph3.Switch("LoadSwitch")
    load_switch.set_parameters(
        three_phase_param(switch_open_resistance),
        three_phase_param(switch_closed_resistance),
        False,
    )

    load_resistor = emt_ph3.Resistor("LoadResistor")
    load_resistor.set_parameters(three_phase_param(load_resistance()))

    slack.connect([n_grid])
    line.connect([n_grid, n_pcc])

    # terminal 0 = GND, terminal 1 = nPcc
    inverter.connect([emt.SimNode.gnd, n_pcc])

    # The resistor load is connected to ground and initially isolated from
    # the PCC by an open switch. Closing the switch creates the disturbance.
    load_switch.connect([n_pcc, n_load])
    load_resistor.connect([n_load, emt.SimNode.gnd])

    system = SystemTopology(
        system_frequency,
        [n_grid, n_pcc, n_load],
        [slack, line, inverter, load_switch, load_resistor],
    )

    system.init_with_powerflow(system_pf, Domain.EMT)

    logger = Logger(sim_name_emt)
    logger.log_attribute("v_pcc", n_pcc.attr("v"))
    logger.log_attribute("i_inv", inverter.attr("i_intf"))
    logger.log_attribute("i_load", load_resistor.attr("i_intf"))
    logger.log_attribute("vc_d", inverter.attr("vc_d"))
    logger.log_attribute("vc_q", inverter.attr("vc_q"))
    logger.log_attribute("p_inst", inverter.attr("p_inst"))
    logger.log_attribute("q_inst", inverter.attr("q_inst"))
    logger.log_attribute("omega_pll", inverter.attr("omega_pll"))

    sim = Simulation(sim_name_emt)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)
    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)

    sim.add_event(dpsimpy.event.SwitchEvent3Ph(load_switch_time, load_switch, True))

    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()


run_emt_simulation(system_pf)

## Build and run the DP-domain dynamic simulation

The DP simulation uses the same topology and parameters as the EMT reference, but with single-phase `DP::Ph1` components throughout, including the DP `AvVoltSourceInverterStateSpace` state-space nodal port under test. `do_system_matrix_recomputation(True)` is required: the inverter is a variable-system-matrix component that relinearizes and restamps every step.

In [ ]:
def run_dp_simulation(system_pf):
    Logger.set_log_dir("logs/" + sim_name_dp)

    n_grid = dp.SimNode("nGrid", PhaseType.Single)
    n_pcc = dp.SimNode("nPcc", PhaseType.Single)
    n_load = dp.SimNode("nLoad", PhaseType.Single)

    slack = dp_ph1.NetworkInjection("Slack")

    line = dp_ph1.PiLine("Line")
    line.set_parameters(
        line_resistance,
        line_inductance,
        line_capacitance,
        line_conductance,
    )

    inverter = dp_ph1.AvVoltSourceInverterStateSpace("INV_SSN_PLL")
    inverter.set_parameters(
        lf,
        cf,
        rf,
        rc,
        system_omega,
        kp_pll,
        ki_pll,
        omega_cutoff,
        p_ref,
        q_ref,
        kp_power_ctrl,
        ki_power_ctrl,
        kp_curr_ctrl,
        ki_curr_ctrl,
    )

    load_switch = dp_ph1.Switch("LoadSwitch")
    load_switch.set_parameters(
        switch_open_resistance,
        switch_closed_resistance,
        False,
    )

    load_resistor = dp_ph1.Resistor("LoadResistor")
    load_resistor.set_parameters(load_resistance())

    slack.connect([n_grid])
    line.connect([n_grid, n_pcc])

    # terminal 0 = GND, terminal 1 = nPcc
    inverter.connect([dp.SimNode.gnd, n_pcc])

    # The resistor load is connected to ground and initially isolated from
    # the PCC by an open switch. Closing the switch creates the disturbance.
    load_switch.connect([n_pcc, n_load])
    load_resistor.connect([n_load, dp.SimNode.gnd])

    system = SystemTopology(
        system_frequency,
        [n_grid, n_pcc, n_load],
        [slack, line, inverter, load_switch, load_resistor],
    )

    system.init_with_powerflow(system_pf, Domain.DP)

    logger = Logger(sim_name_dp)
    logger.log_attribute("v_pcc", n_pcc.attr("v"))
    logger.log_attribute("i_inv", inverter.attr("i_intf"))
    logger.log_attribute("i_load", load_resistor.attr("i_intf"))
    logger.log_attribute("vc_d", inverter.attr("vc_d"))
    logger.log_attribute("vc_q", inverter.attr("vc_q"))
    logger.log_attribute("p_inst", inverter.attr("p_inst"))
    logger.log_attribute("q_inst", inverter.attr("q_inst"))
    logger.log_attribute("omega_pll", inverter.attr("omega_pll"))

    sim = Simulation(sim_name_dp)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.DP)
    sim.set_solver(Solver.MNA)
    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)

    sim.add_event(dpsimpy.event.SwitchEvent(load_switch_time, load_switch, True))

    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()


run_dp_simulation(system_pf)

## Load simulation logs

In [ ]:
pf = pd.read_csv(
    Path("logs") / sim_name_pf / f"{sim_name_pf}.csv",
    skipinitialspace=True,
)

emt_result = pd.read_csv(
    Path("logs") / sim_name_emt / f"{sim_name_emt}.csv",
    skipinitialspace=True,
)

dp_result = pd.read_csv(
    Path("logs") / sim_name_dp / f"{sim_name_dp}.csv",
    skipinitialspace=True,
)

assert len(dp_result) == len(emt_result), (
    "EMT and DP results have a different number of samples; the two time "
    "bases must match for the sample-by-sample tracking checks below."
)

time = time_vector(dp_result)

p_inst_emt = scalar_signal(emt_result, "p_inst")
q_inst_emt = scalar_signal(emt_result, "q_inst")
omega_pll_emt = scalar_signal(emt_result, "omega_pll")
vc_d_emt = scalar_signal(emt_result, "vc_d")
vc_q_emt = scalar_signal(emt_result, "vc_q")
v_pcc_mag_emt = phase_vector_magnitude(signal(emt_result, "v_pcc"))
i_inv_mag_emt = phase_vector_magnitude(signal(emt_result, "i_inv"))
i_load_mag_emt = phase_vector_magnitude(signal(emt_result, "i_load"))

p_inst_dp = scalar_signal(dp_result, "p_inst")
q_inst_dp = scalar_signal(dp_result, "q_inst")
omega_pll_dp = scalar_signal(dp_result, "omega_pll")
vc_d_dp = scalar_signal(dp_result, "vc_d")
vc_q_dp = scalar_signal(dp_result, "vc_q")
v_pcc_mag_dp = envelope_magnitude(complex_signal(dp_result, "v_pcc"))
i_inv_mag_dp = envelope_magnitude(complex_signal(dp_result, "i_inv"))
i_load_mag_dp = envelope_magnitude(complex_signal(dp_result, "i_load"))

print(dp_result.head())

## Parameter exception checks

These checks validate that clearly invalid parameter values are rejected by the DP inverter's `set_parameters`, mirroring the EMT component's own validation.

In [ ]:
valid_parameters = dict(
    lf=lf,
    cf=cf,
    rf=rf,
    rc=rc,
    omega_n=system_omega,
    kp_pll=kp_pll,
    ki_pll=ki_pll,
    omega_cutoff=omega_cutoff,
    p_ref=p_ref,
    q_ref=q_ref,
    kp_power_ctrl=kp_power_ctrl,
    ki_power_ctrl=ki_power_ctrl,
    kp_curr_ctrl=kp_curr_ctrl,
    ki_curr_ctrl=ki_curr_ctrl,
)


def set_inverter_parameters(component, params):
    component.set_parameters(
        params["lf"],
        params["cf"],
        params["rf"],
        params["rc"],
        params["omega_n"],
        params["kp_pll"],
        params["ki_pll"],
        params["omega_cutoff"],
        params["p_ref"],
        params["q_ref"],
        params["kp_power_ctrl"],
        params["ki_power_ctrl"],
        params["kp_curr_ctrl"],
        params["ki_curr_ctrl"],
    )


def expect_parameter_exception(**updates):
    params = valid_parameters.copy()
    params.update(updates)

    component = dp_ph1.AvVoltSourceInverterStateSpace("InvalidParameterTest")

    try:
        set_inverter_parameters(component, params)
    except Exception:
        return

    raise AssertionError(
        f"Expected set_parameters() to raise an exception for updates: {updates}"
    )


expect_parameter_exception(lf=0.0)
expect_parameter_exception(cf=0.0)
expect_parameter_exception(rf=-1.0)
expect_parameter_exception(rc=0.0)
expect_parameter_exception(omega_n=0.0)
expect_parameter_exception(omega_cutoff=-1.0)
expect_parameter_exception(ki_pll=0.0)
expect_parameter_exception(ki_power_ctrl=0.0)
expect_parameter_exception(ki_curr_ctrl=0.0)

print("Parameter exception checks passed.")

## Steady-state checks before and after the load disturbance

The DP simulation is checked in the same two windows as the EMT notebook:

- an initial pre-switch steady-state window,
- a final post-disturbance steady-state window.

Two kinds of checks are applied to the DP result in each window:

- absolute checks against the nominal setpoints, identical to the EMT notebook's own tolerances,
- tracking checks against the EMT reference itself, using the same tolerances, since both simulations use the same time step and can be compared sample by sample.

In [ ]:
initial_window = window(dp_result, 0.02, load_switch_time - 0.01)
final_window = window(dp_result, load_switch_time + 0.5, final_time)

assert initial_window.any()
assert final_window.any()

p_tolerance = 0.01 * abs(p_ref)  # 1 %
q_tolerance = 0.01 * abs(q_ref)  # 1 %
omega_tolerance = 1e-3  # rad/s
vc_q_tolerance = 0.1  # V

# Voltage checks are intentionally formulated as range/settling checks rather
# than direct comparison to nominal voltage, because PCC and filter-side
# voltages include line and coupling voltage drops.
v_pcc_min = 0.85 * grid_voltage_rms_ll
v_pcc_max = 1.10 * grid_voltage_rms_ll
vc_d_min = 0.85 * grid_voltage_rms_ll
vc_d_max = 1.15 * grid_voltage_rms_ll

v_pcc_settling_tolerance = 2.0  # V RMS around window mean
vc_d_settling_tolerance = 2.0  # V RMS around window mean


def assert_window_rms_close(values, target, mask, tolerance, label):
    error = rms(values[mask] - target)
    assert (
        error <= tolerance
    ), f"{label}: RMS error {error:.3e} exceeds tolerance {tolerance:.3e}."


def assert_window_mean_in_range(values, lower, upper, mask, label):
    value = np.mean(values[mask])
    assert lower <= value <= upper, (
        f"{label}: mean value {value:.3e} outside expected range "
        f"[{lower:.3e}, {upper:.3e}]."
    )


def assert_window_settled(values, mask, tolerance, label):
    window_values = values[mask]
    error = rms(window_values - np.mean(window_values))
    assert error <= tolerance, (
        f"{label}: RMS deviation from window mean {error:.3e} exceeds "
        f"tolerance {tolerance:.3e}."
    )


def assert_tracks_reference(values_dp, values_ref, mask, tolerance, label):
    error = rms(values_dp[mask] - values_ref[mask])
    assert error <= tolerance, (
        f"{label}: RMS tracking error {error:.3e} vs. the EMT reference "
        f"exceeds tolerance {tolerance:.3e}."
    )


# Initial steady-state checks, DP vs. nominal setpoints.
assert_window_rms_close(p_inst_dp, p_ref, initial_window, p_tolerance, "initial p_inst")
assert_window_rms_close(q_inst_dp, q_ref, initial_window, q_tolerance, "initial q_inst")
assert_window_rms_close(
    omega_pll_dp, system_omega, initial_window, omega_tolerance, "initial omega_pll"
)
assert_window_rms_close(vc_q_dp, 0.0, initial_window, vc_q_tolerance, "initial vc_q")

assert_window_mean_in_range(vc_d_dp, vc_d_min, vc_d_max, initial_window, "initial vc_d")
assert_window_settled(vc_d_dp, initial_window, vc_d_settling_tolerance, "initial vc_d")

assert_window_mean_in_range(
    v_pcc_mag_dp, v_pcc_min, v_pcc_max, initial_window, "initial PCC voltage magnitude"
)
assert_window_settled(
    v_pcc_mag_dp,
    initial_window,
    v_pcc_settling_tolerance,
    "initial PCC voltage magnitude",
)

# Final post-disturbance steady-state checks, DP vs. nominal setpoints.
assert_window_rms_close(p_inst_dp, p_ref, final_window, p_tolerance, "final p_inst")
assert_window_rms_close(q_inst_dp, q_ref, final_window, q_tolerance, "final q_inst")
assert_window_rms_close(
    omega_pll_dp, system_omega, final_window, omega_tolerance, "final omega_pll"
)
assert_window_rms_close(vc_q_dp, 0.0, final_window, vc_q_tolerance, "final vc_q")

assert_window_mean_in_range(vc_d_dp, vc_d_min, vc_d_max, final_window, "final vc_d")
assert_window_settled(vc_d_dp, final_window, vc_d_settling_tolerance, "final vc_d")

assert_window_mean_in_range(
    v_pcc_mag_dp, v_pcc_min, v_pcc_max, final_window, "final PCC voltage magnitude"
)
assert_window_settled(
    v_pcc_mag_dp,
    final_window,
    v_pcc_settling_tolerance,
    "final PCC voltage magnitude",
)

# DP vs. EMT tracking checks, same tolerances, both windows.
assert_tracks_reference(
    p_inst_dp, p_inst_emt, initial_window, p_tolerance, "initial p_inst DP vs. EMT"
)
assert_tracks_reference(
    q_inst_dp, q_inst_emt, initial_window, q_tolerance, "initial q_inst DP vs. EMT"
)
assert_tracks_reference(
    omega_pll_dp,
    omega_pll_emt,
    initial_window,
    omega_tolerance,
    "initial omega_pll DP vs. EMT",
)
assert_tracks_reference(
    vc_q_dp, vc_q_emt, initial_window, vc_q_tolerance, "initial vc_q DP vs. EMT"
)

assert_tracks_reference(
    p_inst_dp, p_inst_emt, final_window, p_tolerance, "final p_inst DP vs. EMT"
)
assert_tracks_reference(
    q_inst_dp, q_inst_emt, final_window, q_tolerance, "final q_inst DP vs. EMT"
)
assert_tracks_reference(
    omega_pll_dp,
    omega_pll_emt,
    final_window,
    omega_tolerance,
    "final omega_pll DP vs. EMT",
)
assert_tracks_reference(
    vc_q_dp, vc_q_emt, final_window, vc_q_tolerance, "final vc_q DP vs. EMT"
)

# Load current should be almost zero before the switch and nonzero afterwards.
pre_switch_load_window = window(dp_result, 0.02, load_switch_time - 0.01)
post_switch_load_window = window(dp_result, load_switch_time + 0.05, final_time)

assert (
    rms(i_load_mag_dp[pre_switch_load_window]) < 1e-2
), "Load current is not close to zero before the switch event."

assert (
    rms(i_load_mag_dp[post_switch_load_window]) > 1.0
), "Load current is unexpectedly small after the switch event."

print("Steady-state, disturbance-response, and EMT-tracking checks passed.")

## Plot active and reactive power

In [ ]:
plt.figure(figsize=(10, 5))
(p_line,) = plt.plot(time, p_inst_dp, label="p_inst (DP)")
(q_line,) = plt.plot(time, q_inst_dp, label="q_inst (DP)")
plt.plot(
    time, p_inst_emt, label="p_inst (EMT)", linestyle="--", color=p_line.get_color()
)
plt.plot(
    time, q_inst_emt, label="q_inst (EMT)", linestyle="--", color=q_line.get_color()
)
plt.axhline(p_ref, linestyle=":", color=p_line.get_color(), label="p_ref")
plt.axhline(q_ref, linestyle=":", color=q_line.get_color(), label="q_ref")
plt.xlim(0.0, final_time)
plt.ylim(0.0, 2.0 * p_ref)
plt.xlabel("time [s]")
plt.ylabel("power [W / var]")
plt.title("Instantaneous active and reactive power")
plt.grid(True)
plt.legend(loc="best")
plt.tight_layout()
plt.show()

## Plot dq capacitor voltage

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time, vc_d_dp, label="vc_d (DP)")
plt.plot(time, vc_q_dp, label="vc_q (DP)")
plt.plot(time, vc_d_emt, label="vc_d (EMT)", linestyle="--")
plt.plot(time, vc_q_emt, label="vc_q (EMT)", linestyle="--")
plt.xlabel("time [s]")
plt.xlim(0.0, final_time)
plt.ylabel("voltage [V]")
plt.title("Filter capacitor voltage in PLL dq frame")
plt.grid(True)
plt.legend(loc="best")
plt.tight_layout()
plt.show()

## Plot PCC voltage magnitude

The EMT trace is the norm of the three instantaneous abc samples, which is constant for a balanced sinusoid. The DP trace is the corresponding envelope magnitude (see `envelope_magnitude`). Both are on the same scale, but the DP magnitude does not carry the carrier-frequency ripple that a transient abc norm would.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time, v_pcc_mag_dp, label="v_pcc magnitude (DP)")
plt.plot(time, v_pcc_mag_emt, label="v_pcc magnitude (EMT)", linestyle="--")
plt.xlabel("time [s]")
plt.xlim(0.0, final_time)
plt.ylabel("voltage [V]")
plt.title("PCC voltage magnitude")
plt.grid(True)
plt.legend(loc="best")
plt.tight_layout()
plt.show()

## Plot inverter and load current magnitude

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time, i_inv_mag_dp, label="i_inv magnitude (DP)")
plt.plot(time, i_load_mag_dp, label="i_load magnitude (DP)")
plt.plot(time, i_inv_mag_emt, label="i_inv magnitude (EMT)", linestyle="--")
plt.plot(time, i_load_mag_emt, label="i_load magnitude (EMT)", linestyle="--")
plt.xlim(0.0, final_time)
plt.xlabel("time [s]")
plt.ylabel("current [A]")
plt.title("Inverter and switched-load current magnitude")
plt.grid(True)
plt.legend(loc="best")
plt.tight_layout()
plt.show()